PRE-PROCESSING STEPS

In [ ]:
import os
import textract
import PyPDF2
import docx
import pandas as pd
import pytesseract
from PIL import Image
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import inflect
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from scipy.spatial.distance import cosine

# Download necessary nltk data (if needed)
nltk.download('stopwords')
nltk.download('wordnet')

# Set up inflect engine for number-to-word conversion
p = inflect.engine()

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Set Tesseract OCR Path (Windows Only)
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Load RoBERTa model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("C:/Users/Acer/Desktop/ARSwithPredictiveAnalytics/Data-Training/models/RoBERTa-fine-tuned-model")
model = AutoModel.from_pretrained("C:/Users/Acer/Desktop/ARSwithPredictiveAnalytics/Data-Training/models/RoBERTa-fine-tuned-model")

#----------------------------
# TEXT EXTRACTION FUNCTIONS
#----------------------------
def extract_text_from_docx(file_path):
    try:
        doc = docx.Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs])
    except Exception as e:
        print(f"Error extracting text from {file_path} using python-docx: {e}")
        return None

def extract_text_from_pdf(file_path):
    try:
        with open(file_path, "rb") as file:
            reader = PyPDF2.PdfReader(file)
            text = "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])
            return text if text else None
    except Exception as e:
        print(f"Error extracting text from {file_path} using PyPDF2: {e}")
        return None

def extract_text_from_image(file_path):
    try:
        image = Image.open(file_path)
        text = pytesseract.image_to_string(image, config='--psm 6')
        return text.strip() if text else None
    except Exception as e:
        print(f"Error extracting text from {file_path} using OCR: {e}")
        return None

def extract_text(file_path):
    """Detect file type and extract text accordingly."""
    text = None
    if file_path.endswith(".docx"):
        text = extract_text_from_docx(file_path)
    elif file_path.endswith(".pdf"):
        text = extract_text_from_pdf(file_path)
    elif file_path.endswith((".jpg", ".jpeg", ".png", ".JPG")):
        text = extract_text_from_image(file_path)
    
    # Fallback to Textract if primary extraction fails
    if text is None:
        try:
            text = textract.process(file_path).decode("utf-8")
        except Exception as e:
            print(f"Error extracting text from {file_path} using textract: {e}")
            return None
    
    return text

#----------------------------
# TEXT PREPROCESSING
#----------------------------
def preprocess_text(text):
    """Lowercase, remove special characters, normalize numbers, lemmatize, and remove stopwords."""
    if not text:
        return ""

    text = text.lower()
    
    # Normalize numbers (convert digits to words)
    words = text.split()
    normalized_words = []
    for word in words:
        if word.isdigit():  # Check if it's a number
            try:
                word = p.number_to_words(int(word))  # Convert to words
            except:
                word = p.number_to_words(int(word))
                # pass  # If conversion fails, keep the original number
        normalized_words.append(word)
    
    text = " ".join(normalized_words)

    # Remove special characters
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    
    # Lemmatization and Stopword Removal
    text = " ".join([lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words])

    return text

#----------------------------
# RoBERTa SEMANTIC SIMILARITY
#----------------------------
def get_roberta_embedding(text):
    """Get RoBERTa embedding for a given text."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # Use the mean of the last hidden state as the embedding
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embedding

def calculate_semantic_similarity(text1, text2):
    """Calculate cosine similarity between RoBERTa embeddings of two texts."""
    embedding1 = get_roberta_embedding(text1)
    embedding2 = get_roberta_embedding(text2)
    similarity = 1 - cosine(embedding1, embedding2)  # Cosine similarity
    return similarity

#----------------------------
# PROCESS FILES & GENERATE CSV
#----------------------------
def process_files(resume_folder, job_folder, output_csv):
    data = []
    resumes = []
    job_requirements = []
    
    # Extract and preprocess resumes
    for filename in os.listdir(resume_folder):
        file_path = os.path.join(resume_folder, filename)
        if file_path.endswith((".pdf", ".docx", ".jpg", ".jpeg", ".png", ".JPG")):
            print(f"Extracting text from resume: {filename}")
            text = extract_text(file_path)
            if text:
                resumes.append({"filename": filename, "text": preprocess_text(text)})

    # Extract and preprocess job descriptions
    for filename in os.listdir(job_folder):
        file_path = os.path.join(job_folder, filename)
        if file_path.endswith((".pdf", ".docx", ".jpg", ".jpeg", ".png", ".JPG")):
            print(f"Extracting text from job requirement: {filename}")
            text = extract_text(file_path)
            if text:
                job_requirements.append({"filename": filename, "text": preprocess_text(text)})

    # Compute semantic similarity using RoBERTa
    for resume in resumes:
        for job in job_requirements:
            similarity_score = calculate_semantic_similarity(resume["text"], job["text"])
            data.append({
                "Resume Text": resume["text"],
                "Job Text": job["text"],
                "Job Role": job["filename"],
                "Semantic Similarity Score": similarity_score,
            })

    # Save to CSV
    df = pd.DataFrame(data)
    df.to_csv(output_csv, index=False, encoding="utf-8")
    print(f"Extraction complete. Data saved to {output_csv}")

# Example usage
resume_folder = "C:/Users/Acer/Desktop/ARSwithPredictiveAnalytics/Data-Training/all resumes"  # Change this to your resume folder path
job_folder = "C:/Users/Acer/Desktop/ARSwithPredictiveAnalytics/Data-Training/all job requirements"  # Change this to your job requirements folder path
output_csv = "resumes-jobreq-extracted-entity-pairs-roberta.csv"
process_files(resume_folder, job_folder, output_csv)